# Econ 524 — Computational Assignment 1
## OLS, LASSO, and Logistic Regression in High Dimensions



---
### Goal of the assignment

The theory problem set (PS1) asks you to *derive* the properties of these estimators. By the end you should be able to, without looking anything up:

1. **Compute** OLS and LASSO estimators from scratch and with libraries, and know when each is
   appropriate (Q1, Q2).
2. Measure **in-sample vs. out-of-sample** performance correctly, and explain *why* they diverge as
   $p/n$ grows — connecting the simulation to the exact $\sqrt{p/n}$ and $\sqrt{s\log(p\vee n)/n}$
   rates from lecture.
3. Reproduce the **inference failure** of robust standard errors under many covariates, following
   Cattaneo, Jansson & Newey (2018) (Q3).
4. **Tune** a penalty by cross-validation, compare it to the theoretically-motivated choice of
   Belloni, Chen, Chernozhukov & Hansen (2012), and carry the whole toolkit over to
   **classification** via logistic regression (Q2, Q4).


### Rules

- Work in this notebook. Replace every `# YOUR CODE HERE` and every *"Written answer:"* prompt.
- Written answers go in the markdown cells provided; **2–5 sentences each unless stated otherwise**.
  Interpretation is graded as heavily as code — a correct number with no explanation earns half.
- Set a seed wherever there's randomness so your results are reproducible.
- You may use `numpy`, `pandas`, `scipy`, `statsmodels`, `sklearn`. The `hdmpy` package (from W2) is
  optional and only relevant to Q2(e) and Q3(d).
- Don't forget to cite the use of Generative AI. 

### Grade breakdown

| | Topic | Points |
|---|---|---|
| Q1 | OLS: computation & the $p/n$ law, in/out-of-sample | 25 |
| Q2 | LASSO: sparsity, support recovery, tuning, post-LASSO | 30 |
| Q3 | Inference with many covariates (Cattaneo–Jansson–Newey) | 25 |
| Q4 | Logistic regression & high-dimensional classification | 20 |


In [ ]:
# Standard imports for the whole assignment. Add to this cell as needed.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, Lasso, LogisticRegression
from sklearn.model_selection import train_test_split, KFold

import warnings
warnings.simplefilter('ignore')

RNG = np.random.default_rng(524)   # use this generator (or your own seed) throughout

---
# Question 1 — OLS: computation and the $p/n$ law (25 pts)

In `W1_OLS.ipynb` you watched in-sample $R^2$ climb toward 1 as
$p \to n$, even though the data were pure noise ($\beta_0 = 0$). That simulation only looked
**in-sample**. Here you'll (i) build the estimator yourself, (ii) add the out-of-sample side, and
(iii) check the numbers against the exact formulas from Lecture 1.

Recall the two formulas from lecture, for $Y_i = Z_i'\beta_0 + U_i$ with $\mathrm{Var}(U_i)=\sigma^2$:
$$
\mathbb{E}[\text{MSE}_{\text{in}}] = \sigma^2\Big(1 - \tfrac{p}{n}\Big),
\qquad
\mathbb{E}[\text{MSE}_{\text{out}}] = \sigma^2\Big(1 + \tfrac{p}{n-p-1}\Big).
$$


### 1(a) — Compute OLS three ways (5 pts)

Write a function `ols_fit(X, y)` that returns $\hat\beta$ **without** using `statsmodels` or
`sklearn`. Implement it with `np.linalg.lstsq`. Then verify on a small simulated dataset that your
coefficients match `sm.OLS`. Briefly explain why `lstsq` is preferred to
`np.linalg.inv(X.T @ X) @ X.T @ y`.

In [ ]:
def ols_fit(X, y):
    """Return the OLS coefficient vector. X already includes an intercept column if wanted."""
    # YOUR CODE HERE
    raise NotImplementedError

# verification against statsmodels
# YOUR CODE HERE


**Written answer (1a):** *Why `lstsq` over `inv`? (2–3 sentences)*

...

### 1(b) — In- and out-of-sample MSE as a function of $p/n$ (10 pts)

Write `mse_experiment(n, p, sigma=1.0, n_test=5000, reps=200, beta0=None)` that, over `reps`
replications with $\beta_0 = 0$ by default:

- draws a fresh training sample of size $n$ with $p$ standard-normal regressors,
- fits OLS on it (use your `ols_fit`),
- records in-sample MSE and out-of-sample MSE on a fresh test set of size `n_test`,

and returns the **average** of each. Then produce a table over
$p \in \{10, 50, 100, 150, 180\}$ with $n = 200$ that places, side by side, your simulated
$\text{MSE}_{\text{in}}$, $\text{MSE}_{\text{out}}$, and the two theoretical predictions above.

In [ ]:
def mse_experiment(n, p, sigma=1.0, n_test=5000, reps=200, beta0=None, rng=None):
    rng = np.random.default_rng(0) if rng is None else rng
    # YOUR CODE HERE
    raise NotImplementedError

# Build the comparison table for n=200, p in {10,50,100,150,180}
# Columns: p, p/n, MSE_in (sim), MSE_in (theory), MSE_out (sim), MSE_out (theory)
# YOUR CODE HERE


### 1(c) — Plot and interpret (10 pts)

Plot simulated $\text{MSE}_{\text{in}}$ and $\text{MSE}_{\text{out}}$ against $p/n$ on a fine grid
(say $p = 2, 6, 10, \dots, 190$ at $n=200$), overlaying the two theoretical curves. Then answer:

In [ ]:
# YOUR CODE HERE — one figure, both simulated series + both theory curves, axis labels, legend


**Written answers (1c):**

1. *The regressors here are pure noise ($\beta_0 = 0$). At $p/n = 0.5$, how does OLS's
   out-of-sample MSE compare to the trivial "predict 0 for everyone" rule, and what does that say
   about relying on in-sample $R^2$?*

   ...

2. *Lecture gave the prediction-error bound $\sqrt{\mathbb{E}_Z[(Z'\hat\beta_n - Z'\beta_0)^2]}
   \lesssim_p \sqrt{p/n}$. Using your out-of-sample results, argue that this bound captures the
   right rate rather than being loose. (Hint: how does the estimation-error part of $\text{MSE}_{\text{out}}$
   scale with $p/n$ when it is small?)*

   ...

3. **Extension.** *Re-run `mse_experiment` with a sparse truth: `beta0` has 5 entries equal to 1,
   the rest 0. Does the out-of-sample curve still have a minimum at $p = 5$? Why is the "right"
   model no longer obvious just from looking at the data — and how does this motivate LASSO?*

   ...

In [ ]:
# Q1(c) extension: sparse-truth version
# YOUR CODE HERE


---
# Question 2 — LASSO: sparsity, support recovery, and tuning (30 pts)

PS1 Q1(b) contrasts the OLS rate $\sqrt{p/n}$ with the LASSO rate $\sqrt{s\log(p\vee n)/n}$, and
PS1 Q3 asks about robust LASSO and post-LASSO. Here you'll generate that contrast numerically and
examine what LASSO does and does **not** get right about variable selection.

**Data-generating process for this question.** Use a *sparse* linear model:
$$Y_i = Z_i'\beta_0 + U_i, \qquad Z_i \sim N(0, I_p), \quad U_i \sim N(0,1),$$
where only the first $s$ entries of $\beta_0$ are nonzero.

In [ ]:
def make_sparse_data(n, p, s, snr_coefs=None, rng=None):
    """Return X (n,p), y (n,), and the true beta (p,). First s coefs nonzero."""
    rng = np.random.default_rng(0) if rng is None else rng
    beta = np.zeros(p)
    beta[:s] = np.array([3., -2., 2.5, -1.5, 2.]) if snr_coefs is None else snr_coefs
    X = rng.standard_normal((n, p))
    y = X @ beta + rng.standard_normal(n)
    return X, y, beta

### 2(a) — OLS vs LASSO when $p > n$ (7 pts)

Take $n = 150$, $p = 300$, $s = 5$. Try to fit OLS. Then fit `LassoCV`. Report out-of-sample MSE for LASSO on a held-out test set.
Explain what happens to OLS and *why* 

In [ ]:
# n=150, p=300, s=5.  Split, attempt OLS, fit LassoCV, report test MSE.
# YOUR CODE HERE


**Written answer (2a):** *What happens to OLS at $p>n$, and how does LASSO's rate
$\sqrt{s\log(p\vee n)/n}$ explain why LASSO still works here? (3–4 sentences)*

...

### 2(b) — Does LASSO recover the true support? (8 pts)

With $n = 300$, $p = 500$, $s = 5$, fit `LassoCV` and report:

- the set of selected variables (nonzero coefficients),
- whether the true support $\{0,1,2,3,4\}$ is contained in it,
- how many **extra** (false-positive) variables were selected.

Repeat over ~50 simulation replications and report the average number of selected variables and the
fraction of replications in which the true support was fully recovered.

In [ ]:
# YOUR CODE HERE


**Written answer (2b):** *Lecture stated LASSO "tends to select too many regressors (with
small slopes)" and that consistent selection needs a **larger** $\lambda$ than the one optimal for
prediction. Do your results show over-selection? Explain the tension between selecting for
prediction and selecting the exact support. (3–4 sentences)*

...

### 2(c) — The tuning path (5 pts)

For one dataset from 2(b), plot the cross-validation MSE as a function of $\lambda$, marking the selected $\lambda$. On a second plot,
show the number of nonzero coefficients as a function of $\lambda$. Comment on the trade-off.

In [ ]:
# Two plots: (1) CV-MSE vs lambda with chosen lambda marked; (2) # nonzeros vs lambda
# YOUR CODE HERE


**Written answer (2c):** *(2–3 sentences on the trade-off the two plots reveal.)*

...

### 2(d) — CV vs. the theoretical $\lambda$ of Belloni et al. (2012) (6 pts)

Cross-validation is data-driven; BCCH (2012) instead propose a **theory-driven** penalty
$$\lambda = 2c\,\sqrt{n}\,\Phi^{-1}\!\Big(1 - \tfrac{\alpha}{2p}\Big), \qquad c = 1.1,\ \alpha = 0.05,$$
with covariate-specific **penalty loadings** to handle heteroskedasticity (the "robust LASSO").
Compute this $\lambda$ for your 2(b) setting and compare, qualitatively, to the $\lambda$ that
`LassoCV` chose. You do **not** need to match scaling conventions exactly — instead, explain the
conceptual difference in *how* each method picks $\lambda$ and what each is optimizing for.

In [ ]:
from scipy.stats import norm
# Compute the BCCH-style lambda and print alongside LassoCV's chosen alpha.
# (Note: sklearn's objective divides the RSS by 2n, so the scales are not directly comparable —
#  discuss the logic, don't force the numbers to match.)
# YOUR CODE HERE


**Written answer (2d):** *CV picks $\lambda$ to minimize estimated out-of-sample error; BCCH
picks $\lambda$ to dominate the noise with high probability. Explain the difference and when you'd
prefer each. Why does the BCCH $\lambda$ grow with $\log p$ (via $\Phi^{-1}(1-\alpha/2p)$)?
(3–5 sentences)*

...

### 2(e) — Post-LASSO (4 pts)

Implement **post-LASSO** by hand: (1) run LASSO to select variables, (2) run *plain OLS* on only
the selected variables. Compare its out-of-sample MSE to vanilla LASSO on the 2(b) data. Explain
why post-LASSO can reduce the shrinkage bias (PS1 Q3(b)).

*(Optional: if you installed `hdmpy` in W2, compare against `RLasso(post=True)`.)*

In [ ]:
# Post-LASSO by hand: select with LassoCV, refit OLS on selected columns, compare test MSE.
# YOUR CODE HERE


**Written answer (2e):** *Why does re-estimating the selected coefficients by OLS reduce bias
relative to LASSO's shrunk estimates? (2–3 sentences)*

...

---
# Question 3 — Inference with many covariates (25 pts)

This question reproduces the central finding of **Cattaneo, Jansson & Newey (2018)**: when the
number of covariates $p$ is not small relative to $n$, the *usual* heteroskedasticity-robust
standard errors (HC0–HC3) are **inconsistent**, and confidence intervals under-cover. This is the
simulation behind the table on slide 18 of Lecture 1.

**Model.** We care about a single coefficient $\theta_0$ (the "credibility revolution" focus on one
effect):
$$Y_i = \theta_0 D_i + Z_i'\gamma_0 + U_i,$$
with $D_i$ the treatment of interest, $Z_i$ a growing vector of controls, and **heteroskedastic**
errors whose variance depends on $D_i$. We'll set $\theta_0 = 0$ so "coverage" = fraction of
95% CIs that contain 0.

### 3(a) — One coverage simulation (8 pts)

Write `coverage(n, p, reps=1000, theta0=0.0)` that, over `reps` replications:

- draws $D_i \sim N(0,1)$, $Z_i \sim N(0, I_{p-1})$, and $U_i = (0.5 + |D_i|)\,\varepsilon_i$ with
  $\varepsilon_i \sim N(0,1)$ (this is the heteroskedasticity),
- sets $Y_i = \theta_0 D_i + U_i$ (so $\gamma_0 = 0$; controls are irrelevant, which is the cleanest
  case),
- fits OLS of $Y$ on $[D, Z]$ and builds a 95% CI for $\theta_0$ using **HC1** standard errors,
- checks whether the CI covers $\theta_0$.

Return the empirical coverage. You may use `sm.OLS(...).fit(cov_type='HC1')` and read off
`.conf_int()`.

In [ ]:
def coverage(n, p, reps=1000, theta0=0.0, cov_type='HC1', rng=None):
    rng = np.random.default_rng(0) if rng is None else rng
    # YOUR CODE HERE
    raise NotImplementedError

# quick check: at n=800, p=5 coverage should be close to 0.95
# YOUR CODE HERE


### 3(b) — Coverage as $p/n$ grows (9 pts)

Fix $n = 800$. Compute empirical coverage for $p \in \{5, 80, 160, 240, 320\}$, i.e.
$p/n$ from $\approx 0.01$ up to $0.4$. Present a table of coverage vs. $p/n$, and plot it with a
horizontal line at the nominal 0.95.

In [ ]:
# YOUR CODE HERE — table + plot of HC1 coverage vs p/n


**Written answer (3b):** *Describe what happens to coverage as $p/n$ rises, and give the
mechanism. (Hint from lecture: the robust variance uses squared residuals $\hat U_i^2$, but
$\hat U_i = (1-h_{ii})U_i - \sum_{j\ne i} h_{ij}U_j$ and the average leverage is $\bar h = p/n$, so
residuals are systematically too small.) (4–5 sentences)*

...

### 3(c) — Comparing HC0–HC3 (5 pts)

Extend `coverage` to also report HC0, HC2, and HC3. Redo the $p/n$ sweep and put all four in one
table. Which is least bad? Which over-covers?

In [ ]:
# YOUR CODE HERE — coverage table with columns HC0, HC1, HC2, HC3 across p/n


**Written answers (3c):**

1. *Rank HC0–HC3 by how well they cope with large $p/n$, and explain the pattern in terms of how
   each reweights $\hat U_i^2$ by leverage $h_{ii}$. (HC0: no correction; HC1: $n/(n-p)$;
   HC2: $1/(1-h_{ii})$; HC3: $1/(1-h_{ii})^2$.)*

   ...

2. *You'll likely find HC1 $\approx$ HC2 here. Why? (What are the leverages $h_{ii}$ approximately
   equal to when the design is i.i.d. Gaussian?)*

   ...

### 3(d) — The role of heteroskedasticity (3 pts)

Re-run the $p/n$ sweep for HC1 with **homoskedastic** errors ($U_i \sim N(0,1)$, no dependence on
$D_i$). Does the under-coverage largely disappear? Explain why the many-covariates problem for
robust SEs is specifically an *interaction* of high dimension with heteroskedasticity.

*(Cattaneo, Jansson & Newey propose a corrected estimator, sometimes labelled HCK, that remains
valid as $p/n \to \kappa < 1$. Implementing it fully is beyond this assignment, but state in one
sentence what such a fix must do that HC1 does not.)*

In [ ]:
# YOUR CODE HERE — HC1 coverage vs p/n under homoskedasticity


**Written answer (3d):** *(3–4 sentences: role of heteroskedasticity + one sentence on what a
valid many-covariate estimator must correct.)*

...

---
# Question 4 — Logistic regression & high-dimensional classification (20 pts)

Lecture 1 closed on logistic regression as a *linear classification* method, and PS1 Q4 works
through its likelihood and a numeric prediction. Here you'll implement the pieces, verify the PS1
numbers by computation, and carry the regularization story from LASSO over to classification.

### 4(a) — The log-likelihood, by hand (6 pts)

PS1 Q4(a) shows that the per-observation log-likelihood contribution simplifies to
$$\ell_i(\alpha,\beta) = Y_i\{\alpha + X_i'\beta\} - \log\!\big(1 + \exp\{\alpha + X_i'\beta\}\big).$$
Write a function `neg_loglik(params, X, y)` returning the **average negative** log-likelihood
(with `params = [alpha, *beta]`), and minimize it with `scipy.optimize.minimize`. On simulated
binary data, verify your estimates match `sklearn`'s `LogisticRegression(penalty=None)` (or, on
older sklearn, `C=1e10`).

In [ ]:
from scipy.optimize import minimize

def neg_loglik(params, X, y):
    """Average negative log-likelihood. params[0]=alpha, params[1:]=beta. X has NO intercept col."""
    # YOUR CODE HERE
    raise NotImplementedError

# Simulate binary data, minimize neg_loglik, compare to sklearn's (near-)unregularized fit.
# YOUR CODE HERE


**Written answer (4a):** *One sentence on why we minimize the **negative** average
log-likelihood rather than maximize the sum.*

...

### 4(b) — Verify the PS1 numbers (4 pts)

PS1 Q4(b): a fitted logit has $\hat\alpha = -6$, $\hat\beta_1 = 0.05$ (hours studied),
$\hat\beta_2 = 1$ (undergrad GPA). **By computation**, (i) estimate the probability that a student
who studies 40 h with GPA 3.5 earns an A, and (ii) solve for the hours needed for a 50% chance.
Confirm your Q4(a) machinery gives the same probability if you plug in these coefficients.

In [ ]:
# (i) probability at (hours=40, GPA=3.5); (ii) hours for p=0.5
# YOUR CODE HERE


**Written answer (4b):** *State both answers and show the one-line algebra for the 50%
threshold (what does $\alpha + x'\beta = 0$ correspond to?).*

...

### 4(c) — Regularized logistic regression in high dimensions (10 pts)

Lecture 1's last result: unregularized logistic MLE needs $p/n$ small; otherwise regularize with an
$\ell_1$ penalty (the objective on the final lecture slide). Simulate a **sparse, high-dimensional**
classification problem: $n = 200$, $p = 150$, with only the first 5 coefficients nonzero. Compare,
on a held-out test set:

- unregularized logistic regression,
- $\ell_1$-penalized logistic regression (`penalty='l1', solver='saga'`), tuning $C$ by
  cross-validation (`LogisticRegressionCV`),

reporting test accuracy and, for the $\ell_1$ fit, the number of nonzero coefficients recovered.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score

# Simulate sparse high-dim binary data; split; fit both models; report test accuracy + sparsity.
# YOUR CODE HERE


**Written answers (4c):**

1. *Which model predicts better out-of-sample, and why does regularization help here even though it
   biases the coefficients? (Tie this back to the bias–variance trade-off from PS1 Q2(a).)*

   ...

2. *The $\ell_1$ logistic objective and the LASSO objective differ in their first term (log-likelihood
   vs. squared error) but share the $\lambda\|\beta\|_1$ penalty. In one or two sentences, say why the
   **same** penalty produces variable selection in both.*

   ...

---
## Submission checklist

- [ ] Every code cell runs top-to-bottom without error (Kernel → Restart & Run All).
- [ ] Every *"Written answer"* prompt is filled in.
- [ ] Every figure has axis labels and a legend where relevant.
- [ ] Seeds are set so your numbers are reproducible.
- [ ] File renamed to `CA1_<yourlastname>.ipynb`.

**A note on collaboration.** Discussing ideas is fine; the code and written answers you submit must
be your own (course Honor Code applies). If you reuse a block from `W1_OLS.ipynb` or
`W2_Lasso.ipynb`, that's expected and encouraged — no need to cite it. 

### References
- Belloni, A., D. Chen, V. Chernozhukov, and C. Hansen (2012). "Sparse Models and Methods for
  Optimal Instruments with an Application to Eminent Domain." *Econometrica* 80(6), 2369–2429.
- Cattaneo, M. D., M. Jansson, and W. K. Newey (2018). "Inference in Linear Regression Models with
  Many Covariates and Heteroskedasticity." *JASA* 113(523), 1350–1361.
- Chernozhukov et al. (2024). *Applied Causal Inference Powered by ML and AI*, Ch. 1.
